# Etapa B — Par difícil: *Myotis lavali* × *Molossops temminckii*

Par que a chave dos especialistas coloca na mesma faixa de FME (49–52 kHz ×
~50 kHz). Desbalanceamento forte (~13:1). Classe positiva = *temminckii*
(a minoritária), porque é a sensibilidade dela que o Slides_6 diz que cai.

In [1]:
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader

np.random.seed(42)
torch.manual_seed(42)

dados = np.load("../dados_processados/etapa2.npz", allow_pickle=True)

X_todos = dados["X_simples"]
y_especie = dados["y_simples"]
pos_treino = dados["pos_treino_simples"]
pos_val = dados["pos_val_simples"]
pos_teste = dados["pos_teste_simples"]

## Filtrando o par

Mesma divisão da Etapa 2 — só mantém as linhas de cada conjunto que são de
uma das duas espécies. Rótulo: 1 = *temminckii*, 0 = *lavali*.

In [2]:
ESPECIE_POS = "Molossops_temminckii"
ESPECIE_NEG = "Myotis_lavali"


def filtra_par(posicoes):
    pos_par = posicoes[np.isin(y_especie[posicoes], [ESPECIE_POS, ESPECIE_NEG])]
    X = X_todos[pos_par].astype(np.float32)
    y = (y_especie[pos_par] == ESPECIE_POS).astype(np.float32)
    return X, y


X_treino, y_treino = filtra_par(pos_treino)
X_val, y_val = filtra_par(pos_val)
X_teste, y_teste = filtra_par(pos_teste)  # não usar até a Etapa E

for nome, y in [("treino", y_treino), ("validação", y_val), ("teste", y_teste)]:
    print(f"{nome}: {len(y)} gravações, {int(y.sum())} temminckii ({y.mean()*100:.1f}%), "
          f"proporção {int((1-y).sum())}:{int(y.sum())}")

t_X_treino = torch.from_numpy(X_treino)
t_y_treino = torch.from_numpy(y_treino).unsqueeze(1)
t_X_val = torch.from_numpy(X_val)
t_y_val = torch.from_numpy(y_val).unsqueeze(1)

treino: 5489 gravações, 390 temminckii (7.1%), proporção 5099:390
validação: 1177 gravações, 84 temminckii (7.1%), proporção 1093:84
teste: 1175 gravações, 83 temminckii (7.1%), proporção 1092:83


## Os modelos e o treino (mesmas receitas da Etapa A)

- **Linear (A1):** `Linear(5,1)` + sigmoide, `BCELoss`, SGD lr=0,1.
- **MLP (A3):** `Linear(5,8)` + `Tanh` + `Linear(8,1)` + sigmoide, `BCELoss`,
  Adam lr=0,01.

Mini-batch de 64, guardando os pesos do mínimo de validação. MLP com 400 épocas;
**linear com 2000**, porque com 400 o mínimo caiu na última época (ainda não
tinha convergido — SGD puro é mais lento, como visto no A2 × A3).

In [3]:
def cria_linear(n_entradas):
    return nn.Sequential(nn.Linear(n_entradas, 1), nn.Sigmoid())


def cria_mlp(n_entradas, n_ocultos=8):
    return nn.Sequential(
        nn.Linear(n_entradas, n_ocultos), nn.Tanh(),
        nn.Linear(n_ocultos, 1), nn.Sigmoid(),
    )


def treina(modelo, otimizador, criterio, t_Xtr, t_ytr, t_Xva, t_yva, n_epocas=400, semente=42):
    torch.manual_seed(semente)
    loader = DataLoader(TensorDataset(t_Xtr, t_ytr), batch_size=64, shuffle=True)
    perdas_treino, perdas_val = [], []
    melhor, melhores_pesos = float("inf"), None

    for epoca in range(n_epocas):
        modelo.train()
        acumulada = 0.0
        for xb, yb in loader:
            otimizador.zero_grad()
            perda = criterio(modelo(xb), yb)
            perda.backward()
            otimizador.step()
            acumulada += perda.item() * len(xb)
        perdas_treino.append(acumulada / len(t_Xtr))

        modelo.eval()
        with torch.no_grad():
            pv = criterio(modelo(t_Xva), t_yva).item()
        perdas_val.append(pv)
        if pv < melhor:
            melhor, melhores_pesos = pv, copy.deepcopy(modelo.state_dict())

    modelo.load_state_dict(melhores_pesos)
    return perdas_treino, perdas_val


EPOCAS_LINEAR = 2000  # com 400 o linear ainda estava melhorando na última época
EPOCAS_MLP = 400

torch.manual_seed(42)
linear = cria_linear(X_treino.shape[1])
perdas_lin = treina(linear, torch.optim.SGD(linear.parameters(), lr=0.1), nn.BCELoss(),
                    t_X_treino, t_y_treino, t_X_val, t_y_val, n_epocas=EPOCAS_LINEAR)

torch.manual_seed(42)
mlp = cria_mlp(X_treino.shape[1])
perdas_mlp = treina(mlp, torch.optim.Adam(mlp.parameters(), lr=0.01), nn.BCELoss(),
                    t_X_treino, t_y_treino, t_X_val, t_y_val, n_epocas=EPOCAS_MLP)

print(f"Linear — mínimo de validação: {min(perdas_lin[1]):.4f} (época {np.argmin(perdas_lin[1])+1} de {EPOCAS_LINEAR})")
print(f"MLP    — mínimo de validação: {min(perdas_mlp[1]):.4f} (época {np.argmin(perdas_mlp[1])+1} de {EPOCAS_MLP})")

Linear — mínimo de validação: 0.0442 (época 1283 de 2000)
MLP    — mínimo de validação: 0.0148 (época 234 de 400)


## Métricas na validação

Mesmas funções da Etapa A (matriz de confusão e métricas à mão). Aqui, "positivo"
= *temminckii*, então **sensibilidade = fração dos temminckii que o modelo
achou**. Também calculo o "modelo bobo" que sempre responde *lavali*, pra ter
um ponto de comparação pra acurácia.

In [4]:
def matriz_confusao(y_real, y_prob, limiar=0.5):
    y_pred = (y_prob >= limiar).astype(int)
    tp = int(np.sum((y_pred == 1) & (y_real == 1)))
    fp = int(np.sum((y_pred == 1) & (y_real == 0)))
    tn = int(np.sum((y_pred == 0) & (y_real == 0)))
    fn = int(np.sum((y_pred == 0) & (y_real == 1)))
    return tp, fp, tn, fn


def metricas_da_matriz(tp, fp, tn, fn):
    acuracia = (tp + tn) / (tp + fp + tn + fn)
    precisao = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    sensibilidade = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    especificidade = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    f1 = 2 * precisao * sensibilidade / (precisao + sensibilidade) if (precisao + sensibilidade) > 0 else 0.0
    num = (tp * tn) - (fp * fn)
    den = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    mcc = num / den if den > 0 else 0.0
    return {"acurácia": acuracia, "precisão": precisao, "sensibilidade": sensibilidade,
            "especificidade": especificidade, "F1": f1, "MCC": mcc}


def probabilidades(modelo, t_X):
    modelo.eval()
    with torch.no_grad():
        return modelo(t_X).numpy().flatten()


prob_lin_val = probabilidades(linear, t_X_val)
prob_mlp_val = probabilidades(mlp, t_X_val)

cm_lin = matriz_confusao(y_val, prob_lin_val)
cm_mlp = matriz_confusao(y_val, prob_mlp_val)
cm_bobo = matriz_confusao(y_val, np.zeros_like(y_val))  # sempre "lavali"

tabela = pd.DataFrame({
    "sempre lavali": metricas_da_matriz(*cm_bobo),
    "linear": metricas_da_matriz(*cm_lin),
    "MLP": metricas_da_matriz(*cm_mlp),
}).round(4)

print("Matrizes (TP, FP, TN, FN) — positivo = temminckii:")
print(f"  sempre lavali: {cm_bobo}")
print(f"  linear:        {cm_lin}")
print(f"  MLP:           {cm_mlp}")
tabela

Matrizes (TP, FP, TN, FN) — positivo = temminckii:
  sempre lavali: (0, 0, 1093, 84)
  linear:        (78, 7, 1086, 6)
  MLP:           (83, 5, 1088, 1)


,sempre lavali,linear,MLP
acurácia,0.9286,0.9890,0.9949
precisão,0.0000,0.9176,0.9432
sensibilidade,0.0000,0.9286,0.9881
especificidade,1.0000,0.9936,0.9954
F1,0.0000,0.9231,0.9651
MCC,0.0000,0.9171,0.9627


## Checagem: será que o resultado bom demais vem de um atalho?

Hipótese a testar: se *lavali* e *temminckii* foram gravadas em locais/aparelhos
diferentes, as features podem carregar o *setup* de gravação em vez da espécie.
O contexto (área, aparelho, taxa de amostragem) fica só neste diagnóstico —
nunca entra como entrada da rede. Nomes de local trocados por letras.

In [5]:
ctx = pd.read_csv("../dados_processados/contexto_simples.csv")
assert (ctx["Espécie/Sonótipo"].to_numpy() == y_especie).all()  # mesma ordem de X_simples

ctx["split"] = "fora"
ctx.loc[pos_treino, "split"] = "treino"
ctx.loc[pos_val, "split"] = "val"
ctx.loc[pos_teste, "split"] = "teste"

ctx_par = ctx[ctx["Espécie/Sonótipo"].isin([ESPECIE_POS, ESPECIE_NEG])].copy()
ctx_par["Área"] = ctx_par["Área"].fillna("(vazio)")

# anonimiza os locais: A = o mais frequente no par, B = o segundo...
ordem = ctx_par["Área"].value_counts().index
letras = {a: chr(65 + i) if i < 26 else f"Z{i}" for i, a in enumerate(ordem)}
ctx_par["Área_anon"] = ctx_par["Área"].map(letras)

print("Gravações por ÁREA (temminckii × lavali):")
print(pd.crosstab(ctx_par["Área_anon"], ctx_par["Espécie/Sonótipo"]).sort_index())

Gravações por ÁREA (temminckii × lavali):
Espécie/Sonótipo  Molossops_temminckii  Myotis_lavali
Área_anon                                            
A                                    5           1378
B                                   10           1326
C                                    3            748
D                                  378            281
E                                    4            534
F                                   27            465
G                                    1            342
H                                    0            310
I                                    0            308
J                                    5            298
K                                    8            292
L                                    1            219
M                                    3            190
N                                    1            158
O                                    1            155
P                                   11  

In [6]:
for col in ["Modelo_gravador", "Sampling_rate (kHz)", "Marca_gravação", "Tipo_gravação"]:
    print(f"\nGravações por {col}:")
    print(pd.crosstab(ctx_par[col].fillna("(vazio)"), ctx_par["Espécie/Sonótipo"]))


Gravações por Modelo_gravador:
Espécie/Sonótipo  Molossops_temminckii  Myotis_lavali
Modelo_gravador                                      
(vazio)                              0              7
SM2BAT                             557           7277

Gravações por Sampling_rate (kHz):
Espécie/Sonótipo     Molossops_temminckii  Myotis_lavali
Sampling_rate (kHz)                                     
768                                   557           7284

Gravações por Marca_gravação:
Espécie/Sonótipo    Molossops_temminckii  Myotis_lavali
Marca_gravação                                         
Wildlife_acoustics                   557           7284

Gravações por Tipo_gravação:
Espécie/Sonótipo  Molossops_temminckii  Myotis_lavali
Tipo_gravação                                        
Voo_livre                          557           7284


Aparelho, taxa de amostragem e tipo de gravação são idênticos para todo o par
(SM2BAT, 768 kHz, voo livre) — não podem ser o atalho. O **local** é parcialmente
desbalanceado: a maioria dos *temminckii* vem de poucos locais. Mas o local D
também tem muitos *lavali*, então o local sozinho não decide a espécie.

Dois testes pra separar "atalho de local" de "aprendeu o chamado":

1. **Modelo só de local** (sem olhar nenhuma feature acústica): pra cada local,
   prevê a espécie mais comum dele *no treino*. É o quanto o atalho sozinho rende.
2. **Desempenho dentro do local D**, onde as duas espécies convivem em
   quantidade parecida: se o modelo ainda separa bem ali, não é só o local.

In [7]:
area_todas = ctx["Área"].fillna("(vazio)").to_numpy()

def area_do_par(posicoes):
    pos_par = posicoes[np.isin(y_especie[posicoes], [ESPECIE_POS, ESPECIE_NEG])]  # mesma ordem de X_*
    return area_todas[pos_par]

area_treino, area_val = area_do_par(pos_treino), area_do_par(pos_val)

# 1) modelo só de local: espécie mais comum de cada área no treino (empate/desconhecida -> lavali)
prevista_por_area = {}
for a in np.unique(area_treino):
    n_pos = int(((area_treino == a) & (y_treino == 1)).sum())
    n_neg = int(((area_treino == a) & (y_treino == 0)).sum())
    prevista_por_area[a] = 1.0 if n_pos > n_neg else 0.0
pred_local = np.array([prevista_por_area.get(a, 0.0) for a in area_val])

# 2) recortes da validação
area_D_nome = [a for a, letra in letras.items() if letra == "D"][0]
mascara_D = area_val == area_D_nome

def linha(nome, y, prob, mascara=None):
    if mascara is not None:
        y, prob = y[mascara], prob[mascara]
    cm = matriz_confusao(y, prob)
    m = metricas_da_matriz(*cm)
    return {"grupo": nome, "n": len(y), "n temminckii": int(y.sum()), "sensibilidade": round(m["sensibilidade"], 3),
            "precisão": round(m["precisão"], 3), "especificidade": round(m["especificidade"], 3), "MCC": round(m["MCC"], 3)}

tabela_atalho = pd.DataFrame([
    linha("só local — validação toda", y_val, pred_local),
    linha("linear — validação toda", y_val, prob_lin_val),
    linha("MLP — validação toda", y_val, prob_mlp_val),
    linha("só local — dentro do local D", y_val, pred_local, mascara_D),
    linha("linear — dentro do local D", y_val, prob_lin_val, mascara_D),
    linha("MLP — dentro do local D", y_val, prob_mlp_val, mascara_D),
])
tabela_atalho

,grupo,n,n temminckii,sensibilidade,precisão,especificidade,MCC
0,só local — validação toda,1177,84,0.786,0.600,0.960,0.659
1,linear — validação toda,1177,84,0.929,0.918,0.994,0.917
2,MLP — validação toda,1177,84,0.988,0.943,0.995,0.963
3,só local — dentro do local D,99,56,1.000,0.566,0.000,0.000
4,linear — dentro do local D,99,56,0.982,0.982,0.977,0.959
5,MLP — dentro do local D,99,56,1.000,0.982,0.977,0.980


## Teste mais duro: treinar sem o local D e avaliar nele

Mede generalização pra um **local nunca visto** (e quanto a divisão aleatória
por gravação pode estar inflando os números). Treino e checkpoint usam só
linhas de fora do local D; a avaliação usa as linhas do local D que estão em
treino e validação da Etapa 2 (as de **teste ficam de fora**, só entram na
Etapa E).

Ressalva: sem o local D, sobram bem menos *temminckii* pra treinar (a maioria
estava lá). Então uma queda aqui mistura "não generaliza pra local novo" com
"tem menos exemplos" — não dá pra separar os dois só com esse teste.

In [8]:
fora_D_tr = area_treino != area_D_nome
fora_D_va = area_val != area_D_nome

Xtr2, ytr2 = X_treino[fora_D_tr], y_treino[fora_D_tr]
Xva2, yva2 = X_val[fora_D_va], y_val[fora_D_va]

# avaliação: linhas do local D em treino + validação (nunca em teste)
X_D = np.concatenate([X_treino[~fora_D_tr], X_val[~fora_D_va]])
y_D = np.concatenate([y_treino[~fora_D_tr], y_val[~fora_D_va]])

print(f"Treino sem D: {len(ytr2)} linhas, {int(ytr2.sum())} temminckii")
print(f"Validação sem D (checkpoint): {len(yva2)} linhas, {int(yva2.sum())} temminckii")
print(f"Avaliação no local D: {len(y_D)} linhas, {int(y_D.sum())} temminckii")

T = lambda a: torch.from_numpy(a)
tXtr2, tytr2 = T(Xtr2), T(ytr2).unsqueeze(1)
tXva2, tyva2 = T(Xva2), T(yva2).unsqueeze(1)

torch.manual_seed(42)
linear_semD = cria_linear(5)
treina(linear_semD, torch.optim.SGD(linear_semD.parameters(), lr=0.1), nn.BCELoss(),
       tXtr2, tytr2, tXva2, tyva2, n_epocas=EPOCAS_LINEAR)

torch.manual_seed(42)
mlp_semD = cria_mlp(5)
treina(mlp_semD, torch.optim.Adam(mlp_semD.parameters(), lr=0.01), nn.BCELoss(),
       tXtr2, tytr2, tXva2, tyva2, n_epocas=EPOCAS_MLP)

p_lin_D = probabilidades(linear_semD, T(X_D))
p_mlp_D = probabilidades(mlp_semD, T(X_D))

tabela_duro = pd.DataFrame([
    linha("linear (treinado com D) — validação toda", y_val, prob_lin_val),
    linha("MLP (treinado com D) — validação toda", y_val, prob_mlp_val),
    linha("linear SEM D — avaliado em D", y_D, p_lin_D),
    linha("MLP SEM D — avaliado em D", y_D, p_mlp_D),
])
tabela_duro

Treino sem D: 5024 linhas, 128 temminckii
Validação sem D (checkpoint): 1078 linhas, 28 temminckii
Avaliação no local D: 564 linhas, 318 temminckii


,grupo,n,n temminckii,sensibilidade,precisão,especificidade,MCC
0,linear (treinado com D) — validação toda,1177,84,0.929,0.918,0.994,0.917
1,MLP (treinado com D) — validação toda,1177,84,0.988,0.943,0.995,0.963
2,linear SEM D — avaliado em D,564,318,0.925,0.987,0.984,0.902
3,MLP SEM D — avaliado em D,564,318,0.846,0.989,0.988,0.827


## Tratando o desbalanceamento: subamostragem da classe majoritária

**Extra fora dos slides.** A teoria (Medidas de desempenho) só mostra o
problema — acurácia enganosa com classes desbalanceadas — e não ensina
remédios. A subamostragem foi aprovada como experimento pequeno, por não usar
nada novo além de escolher linhas com NumPy.

No **treino**, mantém todos os *temminckii* e sorteia o mesmo número de *lavali*
(proporção 1:1). A **validação não muda** (7% naturais). Repete com 5 sorteios
diferentes pra ver quanto o resultado varia com a amostra escolhida.
O peso por classe na função custo fica de fora por ora (decidir na Etapa C).

In [9]:
def subamostra(X, y, razao=1.0, semente=0):
    """Mantém todos os positivos e sorteia razao*n_pos negativos."""
    rng = np.random.default_rng(semente)
    idx_pos = np.where(y == 1)[0]
    idx_neg = np.where(y == 0)[0]
    escolhidos = rng.choice(idx_neg, size=int(round(len(idx_pos) * razao)), replace=False)
    idx = rng.permutation(np.concatenate([idx_pos, escolhidos]))
    return X[idx], y[idx]


N_SORTEIOS = 5
linhas = []
for semente in range(N_SORTEIOS):
    Xs, ys = subamostra(X_treino, y_treino, razao=1.0, semente=semente)
    tXs, tys = torch.from_numpy(Xs), torch.from_numpy(ys).unsqueeze(1)

    torch.manual_seed(semente)
    lin = cria_linear(5)
    treina(lin, torch.optim.SGD(lin.parameters(), lr=0.1), nn.BCELoss(),
           tXs, tys, t_X_val, t_y_val, n_epocas=EPOCAS_LINEAR, semente=semente)

    torch.manual_seed(semente)
    m = cria_mlp(5)
    treina(m, torch.optim.Adam(m.parameters(), lr=0.01), nn.BCELoss(),
           tXs, tys, t_X_val, t_y_val, n_epocas=EPOCAS_MLP, semente=semente)

    for nome, modelo in [("linear", lin), ("MLP", m)]:
        met = metricas_da_matriz(*matriz_confusao(y_val, probabilidades(modelo, t_X_val)))
        linhas.append({"modelo": nome, "sorteio": semente, **met})

df_sub = pd.DataFrame(linhas)
cols = ["sensibilidade", "precisão", "especificidade", "MCC"]

# referência: mesmo modelo SEM subamostragem (uma execução, treino completo)
ref = pd.DataFrame({
    "linear": metricas_da_matriz(*cm_lin),
    "MLP": metricas_da_matriz(*cm_mlp),
}).T[cols].round(3)
ref.index = [f"{i} — sem subamostragem" for i in ref.index]

sub = df_sub.groupby("modelo")[cols].agg(["mean", "std"]).round(3)
print(f"Com subamostragem 1:1 — média e desvio-padrão em {N_SORTEIOS} sorteios:")
display(sub)
print("Referência (sem subamostragem):")
display(ref)

Com subamostragem 1:1 — média e desvio-padrão em 5 sorteios:


sensibilidade       precisão        especificidade           MCC       
                mean   std     mean    std           mean    std   mean    std
modelo                                                                        
MLP            0.986  0.01    0.889  0.011           0.99  0.001  0.931  0.009
linear         1.000  0.00    0.793  0.042           0.98  0.005  0.881  0.025

Referência (sem subamostragem):


,sensibilidade,precisão,especificidade,MCC
linear — sem subamostragem,0.929,0.918,0.994,0.917
MLP — sem subamostragem,0.988,0.943,0.995,0.963


## Tratando o desbalanceamento: peso por classe (inverso da frequência)

**Extra fora dos slides, mas previsto no plano de trabalho entregue** ("cada exemplo
terá peso inverso à frequência da sua [classe]"). Na função custo, o erro de cada
exemplo é multiplicado pelo peso da sua classe: `n / (2 × n_classe)`, calculado só
no treino. Com 7,1% de *temminckii*, isso dá peso ~7,0 para *temminckii* e ~0,54
para *lavali* (média 1, então a escala da perda não muda).

É parecido com a subamostragem (o modelo passa a "se importar" mais com a classe
rara), mas **sem descartar nenhuma gravação**. A perda de validação usada pra
escolher o checkpoint também é ponderada, pra medir o mesmo que se otimiza. A
avaliação continua na validação natural, sem pesos. 3 repetições (só muda
inicialização e embaralhamento).

In [10]:
def pesos_por_classe(y):
    """Peso inverso à frequência da classe, n / (2 * n_classe): média 1 no treino."""
    n = len(y)
    n_pos = float(y.sum())
    n_neg = n - n_pos
    return n / (2 * n_neg), n / (2 * n_pos)  # (peso da classe 0, peso da classe 1)


def treina_ponderado(modelo, otimizador, t_Xtr, t_ytr, t_Xva, t_yva, n_epocas=400, semente=42):
    w0, w1 = pesos_por_classe(t_ytr.numpy().flatten())
    peso_tr = torch.where(t_ytr == 1, torch.tensor(w1), torch.tensor(w0))
    peso_va = torch.where(t_yva == 1, torch.tensor(w1), torch.tensor(w0))
    criterio = nn.BCELoss(reduction="none")  # perda por exemplo, pra multiplicar pelo peso

    torch.manual_seed(semente)
    loader = DataLoader(TensorDataset(t_Xtr, t_ytr, peso_tr), batch_size=64, shuffle=True)
    melhor, melhores_pesos = float("inf"), None
    for _ in range(n_epocas):
        modelo.train()
        for xb, yb, wb in loader:
            otimizador.zero_grad()
            perda = (criterio(modelo(xb), yb) * wb).mean()
            perda.backward()
            otimizador.step()
        modelo.eval()
        with torch.no_grad():
            pv = (criterio(modelo(t_Xva), t_yva) * peso_va).mean().item()
        if pv < melhor:
            melhor, melhores_pesos = pv, copy.deepcopy(modelo.state_dict())
    modelo.load_state_dict(melhores_pesos)


w0_, w1_ = pesos_por_classe(y_treino)
print(f"Pesos no treino: lavali = {w0_:.3f}, temminckii = {w1_:.3f}")

linhas_w = []
for semente in range(3):
    for nome, criar, otim, n_ep in [
        ("linear", cria_linear, lambda p: torch.optim.SGD(p, lr=0.1), EPOCAS_LINEAR),
        ("MLP", cria_mlp, lambda p: torch.optim.Adam(p, lr=0.01), EPOCAS_MLP),
    ]:
        torch.manual_seed(semente)
        m = criar(5)
        treina_ponderado(m, otim(m.parameters()), t_X_treino, t_y_treino, t_X_val, t_y_val,
                         n_epocas=n_ep, semente=semente)
        cm = matriz_confusao(y_val, probabilidades(m, t_X_val))
        linhas_w.append({"modelo": nome, "semente": semente, "FN": cm[3], "FP": cm[1], **metricas_da_matriz(*cm)})

df_w = pd.DataFrame(linhas_w)
cols_w = ["sensibilidade", "precisão", "especificidade", "MCC"]

comparacao_w = pd.concat({
    "sem correção": pd.DataFrame({"linear": metricas_da_matriz(*cm_lin), "MLP": metricas_da_matriz(*cm_mlp)}).T[cols_w],
    "subamostragem 1:1 (média de 5)": df_sub.groupby("modelo")[cols_w].mean(),
    "peso inverso (média de 3)": df_w.groupby("modelo")[cols_w].mean(),
}, names=["método", "modelo"]).round(3)

print("\nErros médios com peso inverso (FN, FP):")
print(df_w.groupby("modelo")[["FN", "FP"]].mean().round(1))
comparacao_w

Pesos no treino: lavali = 0.538, temminckii = 7.037



Erros médios com peso inverso (FN, FP):
         FN    FP
modelo           
MLP     0.0  10.3
linear  0.0  15.0


sensibilidade  precisão  \
método                         modelo                            
sem correção                   linear          0.929     0.918   
                               MLP             0.988     0.943   
subamostragem 1:1 (média de 5) MLP             0.986     0.889   
                               linear          1.000     0.793   
peso inverso (média de 3)      MLP             1.000     0.891   
                               linear          1.000     0.848   

                                       especificidade    MCC  
método                         modelo                         
sem correção                   linear           0.994  0.917  
                               MLP              0.995  0.963  
subamostragem 1:1 (média de 5) MLP              0.990  0.931  
                               linear           0.980  0.881  
peso inverso (média de 3)      MLP              0.991  0.939  
                               linear           0.986  0.915

## Representação completa × simples (mesmas gravações)

A completa oficial tem **25 colunas**: 3 pulsos × 5 descritores, média e
desvio-padrão (o tipo de pulso ficou de fora, ver a seção seguinte). Ela só existe
pras gravações de espécie única com exatamente 3 pulsos. Pra comparar de forma
justa, treino e avalio **as duas representações nas mesmas gravações** (as que
estão nas duas), com a mesma divisão da Etapa 2.
Mesmas receitas de treino (linear 2000 épocas, MLP 400, 8 neurônios ocultos);
3 repetições variando só a inicialização e o embaralhamento, pra ver o quanto o
resultado oscila.

In [11]:
arq_completa = pd.read_csv("../dados_processados/arquivos_completa.csv")["Arquivo_norm"].to_numpy()
pos_na_completa = {a: i for i, a in enumerate(arq_completa)}
X_completa = dados["X_completa"]  # 25 colunas, sem tipo de pulso
arq_simples = ctx["Arquivo_norm"].to_numpy()  # mesma ordem de X_simples

pos_c = {"treino": dados["pos_treino_completa"], "val": dados["pos_val_completa"], "teste": dados["pos_teste_completa"]}


def linhas_comuns(posicoes, nome_split):
    """Linhas do par (em `posicoes`, da simples) que também estão na completa."""
    pos_par = posicoes[np.isin(y_especie[posicoes], [ESPECIE_POS, ESPECIE_NEG])]
    tem = np.array([arq_simples[p] in pos_na_completa for p in pos_par])
    pos_par = pos_par[tem]
    idx_c = np.array([pos_na_completa[arq_simples[p]] for p in pos_par])
    # a divisão da completa foi herdada da simples: confere que o split bate
    assert np.isin(idx_c, pos_c[nome_split]).all()
    y = (y_especie[pos_par] == ESPECIE_POS).astype(np.float32)
    return X_todos[pos_par].astype(np.float32), X_completa[idx_c].astype(np.float32), y, idx_c


Xs_tr, Xc_tr, yc_tr, idx_tr = linhas_comuns(pos_treino, "treino")
Xs_va, Xc_va, yc_va, idx_va = linhas_comuns(pos_val, "val")
print(f"Gravações em comum — treino: {len(yc_tr)} ({int(yc_tr.sum())} temminckii), "
      f"validação: {len(yc_va)} ({int(yc_va.sum())} temminckii)")
print(f"Colunas — simples: {Xs_tr.shape[1]}, completa: {Xc_tr.shape[1]}")

T = lambda a: torch.from_numpy(a)
tyc_tr, tyc_va = T(yc_tr).unsqueeze(1), T(yc_va).unsqueeze(1)

N_REP = 3
achados = []
for rep in range(N_REP):
    for nome_rep, Xtr_, Xva_ in [("simples", Xs_tr, Xs_va), ("completa (25 col)", Xc_tr, Xc_va)]:
        n_in = Xtr_.shape[1]
        tXtr_, tXva_ = T(Xtr_), T(Xva_)

        torch.manual_seed(rep)
        lin = cria_linear(n_in)
        treina(lin, torch.optim.SGD(lin.parameters(), lr=0.1), nn.BCELoss(),
               tXtr_, tyc_tr, tXva_, tyc_va, n_epocas=EPOCAS_LINEAR, semente=rep)

        torch.manual_seed(rep)
        m = cria_mlp(n_in)
        treina(m, torch.optim.Adam(m.parameters(), lr=0.01), nn.BCELoss(),
               tXtr_, tyc_tr, tXva_, tyc_va, n_epocas=EPOCAS_MLP, semente=rep)

        for nome_mod, modelo in [("linear", lin), ("MLP", m)]:
            cm = matriz_confusao(yc_va, probabilidades(modelo, tXva_))
            achados.append({"representação": nome_rep, "modelo": nome_mod, "rep": rep,
                            "FN": cm[3], "FP": cm[1], **metricas_da_matriz(*cm)})

df_rep = pd.DataFrame(achados)
cols = ["FN", "FP", "sensibilidade", "precisão", "MCC"]
comparacao = df_rep.groupby(["modelo", "representação"])[cols].agg(["mean", "std"]).round(3)
comparacao

Gravações em comum — treino: 5357 (353 temminckii), validação: 1148 (75 temminckii)
Colunas — simples: 5, completa: 25


FN            FP        sensibilidade         \
                           mean    std   mean    std          mean    std   
modelo representação                                                        
MLP    completa (25 col)  0.333  0.577  2.000  1.000         0.996  0.008   
       simples            1.667  0.577  3.333  2.082         0.978  0.008   
linear completa (25 col)  2.000  0.000  4.000  0.000         0.973  0.000   
       simples            5.000  0.000  4.000  0.000         0.933  0.000   

                         precisão           MCC         
                             mean    std   mean    std  
modelo representação                                    
MLP    completa (25 col)    0.974  0.013  0.984  0.008  
       simples              0.957  0.026  0.965  0.012  
linear completa (25 col)    0.948  0.000  0.958  0.000  
       simples              0.946  0.000  0.935  0.000

## Diagnóstico: por que o tipo de pulso ficou de fora da completa

Na primeira versão a completa incluía 4 colunas one-hot de tipo de pulso, e o
resultado foi **perfeito demais** (0 erros, MCC 1,000, nos dois modelos). A
tabela abaixo mostra por quê: cada tipo de pulso corresponde a um grupo fixo de
espécies, **sem nenhuma exceção**. Depois se confirmou (24/09) que, na planilha
original, essa coluna é uma **fórmula do Excel que devolve o tipo a partir do nome
da espécie**: não é medida da gravação, é um rótulo disfarçado (vazamento). Esta
versão com tipo é só diagnóstico e **nunca entra como entrada** dos modelos
oficiais.

In [12]:
X_ct = dados["X_completa_com_tipo"]  # 29 colunas — só diagnóstico
cols_ct = list(dados["colunas_completa_com_tipo"])
i_tipo = [i for i, c in enumerate(cols_ct) if c.startswith("tipo_")]

# 1) tipo de pulso x espécie (volta à escala original pra ler o one-hot)
Xo = X_ct * dados["desvio_completa_com_tipo"] + dados["media_completa_com_tipo"]
tipo_nome = np.array([cols_ct[i][5:] for i in i_tipo])[np.argmax(Xo[:, i_tipo], axis=1)]
print("Tipo de pulso × espécie (as 8 espécies, representação completa):")
print(pd.crosstab(pd.Series(dados["y_completa"], name="espécie"), pd.Series(tipo_nome, name="tipo de pulso")))

# 2) MLP (3 repetições) nas mesmas gravações: com o tipo, e só com o tipo
diag = []
for nome_cfg, usar in [("completa + tipo (29 col)", list(range(len(cols_ct)))),
                       ("SÓ tipo de pulso (4 col)", i_tipo)]:
    Xtr_ = X_ct[idx_tr][:, usar].astype(np.float32)
    Xva_ = X_ct[idx_va][:, usar].astype(np.float32)
    for rep in range(N_REP):
        torch.manual_seed(rep)
        m = cria_mlp(len(usar))
        treina(m, torch.optim.Adam(m.parameters(), lr=0.01), nn.BCELoss(),
               T(Xtr_), tyc_tr, T(Xva_), tyc_va, n_epocas=EPOCAS_MLP, semente=rep)
        cm = matriz_confusao(yc_va, probabilidades(m, T(Xva_)))
        diag.append({"configuração (MLP)": nome_cfg, "FN": cm[3], "FP": cm[1], **metricas_da_matriz(*cm)})

print("\nMLP com o tipo de pulso (diagnóstico), média de 3 repetições:")
pd.DataFrame(diag).groupby("configuração (MLP)")[["FN", "FP", "sensibilidade", "precisão", "MCC"]].mean().round(3)

Tipo de pulso × espécie (as 8 espécies, representação completa):
tipo de pulso           FMa/qCF  FMa/qCFd  FMd/qCFd  qCFd
espécie                                                  
Eumops_sp                     0         0         0   685
Lasiurus_blossevillii         0         0      1006     0
Molossops_temminckii        502         0         0     0
Molossus_currentium           0       638         0     0
Molossus_molossus             0      1559         0     0
Myotis_lavali                 0         0      7154     0
Myotis_nigricans              0         0       370     0
Neoeptesicus_furinalis        0         0       952     0



MLP com o tipo de pulso (diagnóstico), média de 3 repetições:


,FN,FP,sensibilidade,precisão,MCC
configuração (MLP),,,,,
SÓ tipo de pulso (4 col),0.0,0.0,1.0,1.0,1.0
completa + tipo (29 col),0.0,0.0,1.0,1.0,1.0
